# Import des librairies

In [69]:
import re
import numpy as np
from dataclasses import dataclass

# Definition classe

In [70]:
@dataclass
class Light:
    col: int
    row: int
    statut: str
    _pos: tuple[int,int] = None


    @property
    def pos(self):
        if self._pos is None:
            self._pos = [self.row, self.col]
        return self._pos

# Nettoyage des données

In [71]:
with open("real.txt") as file:
    grid = np.array([
        [e for e in row]
        for row in file.read().splitlines()
    ])

dim_grid = len(grid)

grid

array([['#', '.', '.', ..., '#', '.', '.'],
       ['#', '#', '#', ..., '#', '.', '#'],
       ['.', '.', '.', ..., '.', '#', '.'],
       ...,
       ['#', '.', '.', ..., '.', '.', '.'],
       ['#', '#', '#', ..., '.', '#', '.'],
       ['.', '#', '.', ..., '#', '#', '.']], shape=(100, 100), dtype='<U1')

# Allume les corners

In [72]:
for num_row in [0,dim_grid-1]:
    for num_col in [0,dim_grid-1]:
        grid[num_row, num_col] = "#"
grid

array([['#', '.', '.', ..., '#', '.', '#'],
       ['#', '#', '#', ..., '#', '.', '#'],
       ['.', '.', '.', ..., '.', '#', '.'],
       ...,
       ['#', '.', '.', ..., '.', '.', '.'],
       ['#', '#', '#', ..., '.', '#', '.'],
       ['#', '#', '.', ..., '#', '#', '#']], shape=(100, 100), dtype='<U1')

# Détermine si une lumière se situe dans un angle

In [73]:
def func_light_is_in_corner(
    in_light: Light, 
    in_dim_grid: int
) -> bool: 
    
    # Coin en haut à gauche/droite
    if in_light.row == 0 and (in_light.col == 0 or in_light.col == in_dim_grid-1):
        return True
    
    # Coins en bas à gauche/doite
    if in_light.row == in_dim_grid-1 and (in_light.col == 0 or in_light.col == in_dim_grid-1):
        return True

    return False   

# Détermine pour une lumière si on doit la switch

In [74]:
def func_get_new_statut_light(
    in_light: Light,
    in_grid: np.array,
    in_dim_grid: int
):
    n_neighbors = 0

    range_row = range(max(in_light.row-1, 0), min(in_light.row+2, in_dim_grid))
    range_col = range(max(in_light.col-1, 0), min(in_light.col+2, in_dim_grid))

    # Pour chaque voisins
    for row_neighbor in range_row:
        for col_neighbor in range_col:

            # Créer un objet Light
            neighbor = Light(
                col=col_neighbor,
                row=row_neighbor,
                statut=in_grid[row_neighbor,col_neighbor]
            )

            # Ne pas explorer la positon actuelle
            if neighbor.pos == in_light.pos:
                continue
            
            # Si la lumière du voisin est allumée
            if neighbor.statut == "#":

                # On incrémente le compteur
                n_neighbors += 1
    
    # Si la lumière orinale est allumée
    if in_light.statut == "#": 

        # Si le nombre de voisins est compris entre 2 et 3
        if 2 <= n_neighbors <= 3:
            
            # On ne fais rien
            return None

        # Si le nombre de voisins n'est pas compris entre 2 et 3
        else:
            # On l'éteint si la 
            in_light.statut = "." if not func_light_is_in_corner(in_light = in_light, in_dim_grid = in_dim_grid) else "#"
    
    # Si la lumière originale est éteinte
    else:

        # Si elle a exactement 3 voisins
        if n_neighbors == 3:

            # On l'allume
            in_light.statut = "#"
        
        # Si le nombre de voisins est différent de 3
        else:

            # On ne fais rien
            return None
        
    return in_light


# Récupère les lumières à switch

In [75]:
def func_get_lights_to_switch(
    in_grid: np.array,
    in_dim_grid: int
) -> list[Light]:
    
    lights_to_switch = [
        res
        for num_row, row in enumerate(in_grid)
        for num_col, e in enumerate(row)
        if (res:= func_get_new_statut_light(
                in_dim_grid=in_dim_grid,
                in_grid=in_grid,
                in_light=Light(
                    col= num_col,
                    row = num_row,
                    statut = e
                )
            )) is not None
    ]

    return lights_to_switch

# Switch le statut des lumières récupérés

In [76]:
def func_switch_all_lights_statut(
    in_grid: np.array,
    in_lights_to_switch: list[Light]
) -> np.array:
    
    # Pour chaque lumière à switch
    for light_to_switch in in_lights_to_switch:

        # On switch son statut (on/off)
        in_grid[light_to_switch.row, light_to_switch.col]=light_to_switch.statut
    
    return in_grid

# func_switch_all_lights_statut(in_grid = data, in_lights_to_switch=lights_to_switch)

# Execute une étape

In [77]:
def func_execute_step(
    in_grid: np.array,
    in_dim_grid: int
):
    
    lights_to_switch = func_get_lights_to_switch(
        in_grid = in_grid,
        in_dim_grid = in_dim_grid    
    )

    new_grid = func_switch_all_lights_statut(
        in_grid = in_grid,
        in_lights_to_switch = lights_to_switch    
    )
    
    return new_grid

# Execute toutes les étapes

In [78]:
print(grid)
for i in range(100):
    grid = func_execute_step(
        in_grid = grid,
        in_dim_grid = dim_grid
    )
    # print(grid)
    # print()

sum(
    1 
    for row in grid    
    for light in row
    if light == "#"
 )

[['#' '.' '.' ... '#' '.' '#']
 ['#' '#' '#' ... '#' '.' '#']
 ['.' '.' '.' ... '.' '#' '.']
 ...
 ['#' '.' '.' ... '.' '.' '.']
 ['#' '#' '#' ... '.' '#' '.']
 ['#' '#' '.' ... '#' '#' '#']]


924